# Load the data

In [1]:
import pandas as pd

#Load datasets
reviews = pd.read_csv("../data/reviews83325.csv")
places = pd.read_csv("../data/Tripadvisor.csv")

print("Reviews shape:", reviews.shape)
print("Places shape:", places.shape)

C:\Users\geoff\AppData\Local\Temp\ipykernel_43148\797032139.py:4: DtypeWarning: Columns (15,16,17,18,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews = pd.read_csv("../data/reviews83325.csv")


Reviews shape: (340385, 21)
Places shape: (3761, 60)


In [11]:
reviews.head(2)


,id,idplace,titre,idauteur,review,note,date_review,date_visit,langue,published_platform,...,subratings,machine_translated,machine_translatable,owner_id,owner_langue,owner_date_review,owner_connection,owner_responder,owner_response,owner_title
0,771569620,188467,February,F645CC9429E8A40EB1F5A487780EC683,Personally I think it is the most beautiful sq...,5,23/9/2020 11:14:11,2020-02,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,769814072,188467,Nice green square,AFFB511F21DF819776CB2F8013034382,We walked through this lovely park but did not...,4,11/9/2020 07:52:32,2019-10,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 340385 entries, 0 to 340384
Data columns (total 21 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    340385 non-null  int64  
 1   idplace               340385 non-null  int64  
 2   titre                 340382 non-null  object 
 3   idauteur              339726 non-null  object 
 4   review                340385 non-null  object 
 5   note                  340385 non-null  int64  
 6   date_review           340385 non-null  object 
 7   date_visit            321147 non-null  object 
 8   langue                340385 non-null  object 
 9   published_platform    340385 non-null  object 
 10  typeReview            340385 non-null  object 
 11  subratings            340385 non-null  object 
 12  machine_translated    340385 non-null  bool   
 13  machine_translatable  340385 non-null  bool   
 14  owner_id              39641 non-null   float64
 15  

In [13]:
places.head(2)

,id,idTrip,fromId,nom,url,rating,nbAvis,nbAvisRecupere,latitude,longitude,...,ap_exclusion,ap_inclusions,ap_introduction,ap_primary_supplier_attraction_id,ap_primary_supplier_subtype,ap_primary_ta_geo_id,ap_product_code,ap_product_highlights,ap_product_text,ap_raw
0,188467,187147.0,187070.0,Place des Vosges,https://www.tripadvisor.fr/Attraction_Review-g...,4.108407,5663,5664.0,48.855614,2.365553,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,188468,187147.0,187070.0,Rue des Francs Bourgeois,https://www.tripadvisor.fr/Attraction_Review-g...,3.316532,73,73.0,48.858140,2.359880,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
places.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3761 entries, 0 to 3760
Data columns (total 60 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   id                                 3761 non-null   int64  
 1   idTrip                             3754 non-null   float64
 2   fromId                             201 non-null    float64
 3   nom                                3761 non-null   object 
 4   url                                3697 non-null   object 
 5   rating                             3661 non-null   float64
 6   nbAvis                             3761 non-null   int64  
 7   nbAvisRecupere                     2344 non-null   float64
 8   latitude                           3761 non-null   float64
 9   longitude                          3761 non-null   float64
 10  typeR                              3761 non-null   object 
 11  adresse                            1028 non-null   objec

# Filter English Reviews Only

In [16]:
reviews_en = reviews[reviews["langue"] == "en"]
print("English reviews shape:", reviews_en.shape)

English reviews shape: (153071, 21)


# Merge Reviews with Places

we join: 
- reviews.idplace
- places.id

In [18]:
data = reviews_en.merge(places,
                        left_on="idplace",
                        right_on="id",
                        how="inner")

print("Merged dataset shape:", data.shape)
                        

Merged dataset shape: (153071, 81)


Now each review is linked to its place information

Need to check if this merge actually works well or if some data is lost 

In [21]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 153071 entries, 0 to 153070
Data columns (total 81 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   id_x                               153071 non-null  int64  
 1   idplace                            153071 non-null  int64  
 2   titre                              153070 non-null  object 
 3   idauteur                           153061 non-null  object 
 4   review                             153071 non-null  object 
 5   note                               153071 non-null  int64  
 6   date_review                        153071 non-null  object 
 7   date_visit                         142576 non-null  object 
 8   langue                             153071 non-null  object 
 9   published_platform                 153071 non-null  object 
 10  typeReview                         153071 non-null  object 
 11  subratings                         1530

# Aggregate Reviews by Place

we want one document per place


Need to change this, so it includes commas to be able to select reviews, or remove and do later after review selection

In [22]:
grouped = data.groupby("idplace")["review"].apply(lambda x: " ".join(x)).reset_index()
grouped.columns = ["place_id", "full_text_reviews"]

print("Number of uniques places:", len(grouped))
grouped.head()

Number of uniques places: 1835


,place_id,full_text_reviews
0,188467,Personally I think it is the most beautiful sq...
1,188468,My old college friend and I booked this beauti...
2,188470,"Being winter and all, not a lot going on, howe..."
3,188471,To call Au Passe Partout a shop is a serious u...
4,188472,Very old historical place. I attended to exper...


Now:
- One row = One place
- All reviews concatenated into one large text

# Add metadata (For Evaluation Only)

we keep metadata for later evaluation (NOT for model training)

In [26]:
metadata_cols = [
    "id",
    "typeR",
    "activiteSubCategorie",
    "activiteSubType",
    "restaurantType",
    "restaurantTypeCuisine",
    "priceRange"
]

metadata = places[metadata_cols]

final_dataset = grouped.merge(
    metadata,
    left_on="place_id",
    right_on="id",
    how="left"
)

print("Final dataset shape:", final_dataset.shape)
final_dataset.head()

Final dataset shape: (1835, 9)


,place_id,full_text_reviews,id,typeR,activiteSubCategorie,activiteSubType,restaurantType,restaurantTypeCuisine,priceRange
0,188467,Personally I think it is the most beautiful sq...,188467,A,47,163,NaN,NaN,NaN
1,188468,My old college friend and I booked this beauti...,188468,A,47,163,NaN,NaN,NaN
2,188470,"Being winter and all, not a lot going on, howe...",188470,A,"26,47,51","34,144",NaN,NaN,NaN
3,188471,To call Au Passe Partout a shop is a serious u...,188471,A,26,137,NaN,NaN,NaN
4,188472,Very old historical place. I attended to exper...,188472,A,47,10,NaN,NaN,NaN


# Quick Exploratory Analysis

Distribution of places types

In [28]:
print(final_dataset["typeR"].value_counts())

typeR
AP    989
R     538
A     252
H      56
Name: count, dtype: int64


Text length statistics

In [29]:
final_dataset["text_length"] = final_dataset["full_text_reviews"].apply(lambda x: len(x.split()))
print(final_dataset["text_length"].describe())

count    1.835000e+03
mean     6.677538e+03
std      4.871902e+04
min      1.700000e+01
25%      1.505000e+02
50%      5.750000e+02
75%      2.763000e+03
max      1.896322e+06
Name: text_length, dtype: float64


This will show:
- Minimum number of words
- Maximum number of words
- Average document length